# Lab 1: Agents & Models

**Difficulty: Beginner | ~30 min | No prerequisites**

You will build a tiny AI agent, give it a tool, and swap the model underneath it — proving the model is a swappable part, not the whole system.

## Step 1 — Load the key

This cell loads the `.env` file and fails loudly if your key is missing — better to find out here than halfway through.

In [ ]:
import os
from dotenv import load_dotenv

# Read the OPENROUTER_API_KEY we saved in .env (Step 9 of the guide)
load_dotenv()

# Stop early with a clear message if the key is missing
if not os.getenv("OPENROUTER_API_KEY"):
    raise SystemExit("No OPENROUTER_API_KEY found. Add it to .env and restart the kernel.")

## Step 2 — Initialize a model

`ChatOpenAI` is LangChain's wrapper for any OpenAI-compatible API, and OpenRouter is one. `model=` is the free model ID, `base_url=` points at OpenRouter, and `temperature=0` keeps answers factual.

The key comes from the environment variable we just loaded — never hardcode a secret (Article CQ-7).

In [ ]:
from langchain_openai import ChatOpenAI

# Model 1: an open-weight model served free by OpenRouter
# base_url redirects the standard OpenAI client to OpenRouter
model_1 = ChatOpenAI(
    model="openai/gpt-oss-20b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)

## Step 3 — Write a tool

A tool is just a function with a clear docstring and typed arguments. LangChain reads the docstring to tell the model what the tool does.

In [ ]:
def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return their product."""
    return a * b

## Step 4 — Create the agent

`create_agent` wraps the model and tools into the agent loop. One line.

In [ ]:
from langchain.agents import create_agent

# The agent = this model + this tool + the decision loop around them
agent = create_agent(model_1, tools=[multiply])

## Step 5 — Ask a question that needs the tool

`invoke` runs the whole loop: the agent decides it needs math, calls `multiply(8, 7)`, gets `56` back, and answers. The last message in `messages` is the final answer.

In [ ]:
# Run the agent loop with a user message
result_1 = agent.invoke({
    "messages": [("user", "What is 8 multiplied by 7?")],
})

# Show only the final answer, not the whole conversation
result_1["messages"][-1].content

## Step 6 — Swap the model

Build a *second* agent with a different free model. Same tool, same structure — **only the model changed.** This is the whole point of the lab.

In [ ]:
# Model 2: a different open-weight model, still free on OpenRouter
model_2 = ChatOpenAI(
    model="nvidia/nemotron-nano-9b-v2:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)

# Same tools, same loop, new brain
agent_2 = create_agent(model_2, tools=[multiply])

## Step 7 — Ask the same question with the new brain

Identical question, identical agent structure. Only the model differs.

In [ ]:
# Run the same question through the swapped model
result_2 = agent_2.invoke({
    "messages": [("user", "What is 8 multiplied by 7?")],
})
result_2["messages"][-1].content

## Step 8 — A question with no tool needed

Agents don't always use tools. This question is answered directly from the model's knowledge.

In [ ]:
# A plain-knowledge question: no tool, the loop just answers
result_3 = agent_2.invoke({
    "messages": [("user", "In one sentence, what is an AI agent?")],
})
result_3["messages"][-1].content

## Optional Exercise

Swap in a third model. Add a new cell: create a `ChatOpenAI` with the free model `inclusionai/ling-3.0-flash:free`, build an `agent_3` with the same `multiply` tool, and ask it the same "8 multiplied by 7" question. Confirm it returns 56. If that model ID no longer exists, pick any `:free` model at https://openrouter.ai/models.